# Babamul stream

Read alerts and do inference using a kafka topic and babamul

Copied directly from: https://github.com/boom-astro/babamul/blob/main/examples/stream-basic/notebook.ipynb

In [ ]:
#imports 
import numpy as np
from astropy.time import Time
import yaml
import requests

import dotenv
from astropy.coordinates import SkyCoord
from astropy.table import Table
from tqdm.notebook import tqdm

import babamul
from babamul import LsstAlert, ZtfAlert, jupyter

dotenv.load_dotenv()

In [ ]:
# listen to all Rubin alerts with no ZTF detection, hostless
# as we could expect for distant transients

topics = ["babamul.lsst.no-ztf-match.hostless"]

In [ ]:
# first, let's just grab one alert and inspect it
# Note: initializing a consumer can take a few seconds, based on your distance
#       from the Kafka cluster and the number of topics you subscribe to. Once
# .      alerts start flowing, things get a lot faster!
with babamul.AlertConsumer(
    topics=topics,
    offset="earliest",
    auto_commit=False,
    timeout=15,
) as consumer:
    for alert in consumer:
        alert.show()
        break

In [ ]:
# apply generic filters on alerts before doing the crossmatch
# https://sdm-schemas.lsst.io/apdb.html

def is_relevant(alert: ZtfAlert | LsstAlert):
    if isinstance(alert, ZtfAlert):
        return False # for now we read from the no ztf match topic
    if isinstance(alert, LsstAlert):
        if alert.candidate.isDipole: # 	Source well fit by a dipole, ie image subtraction artifact
            return False
        if alert.candidate.psfFlux_flag: # Failure to derive linear least-squares fit of psf model. 
            return False
        # Only consider alerts with a reasonable PSF fit (using a threshold on the reduced chi2 of the PSF fit)
        if alert.candidate.psfChi2 / alert.candidate.psfNdata > 10.0:
            return False
        if alert.candidate.snr < 3.0:
            return False
        if alert.candidate.extendedness is None: #0-1 where 1=extended.
            return False
        if alert.candidate.shape_flag: # set if anything went wrong when measuring the shape
            return False
        if alert.candidate.centroid_flag: # set if anything went wrong when fitting the centroid
            return False

    # Only consider alerts with a real-bogus score above 0.4
    if alert.drb < 0.4:
        return False

    # Exclude alerts that are likely to be known SSOs
    if alert.properties.rock:
        return False

    # Exclude alerts that are likely to be stars or near bright stars
    if alert.properties.star:  # using PS1 PSC for ZTF, using LSPSC for LSST
        return False

    if alert.properties.near_brightstar:  # same here
        return False

    # Only consider positive subtractions (i.e. candidate is brighter than the reference)
    if not alert.candidate.isdiffpos:
        return False

    # Filter for bright mag in PSF photometry - currently leave, as our source are dim
    # if alert.candidate.magpsf is None or alert.candidate.magpsf > 21.5:
    #     return False

    # Exclude alerts that are not "stationary", i.e. not detected at least twice with sufficient time separation
    # (helps rule out uncataloged asteroids, some boguses, ...)
    return alert.properties.stationary

In [ ]:
# steam 2000 alerts

alerts: list[babamul.ZtfAlert | babamul.LsstAlert] = []
limit = 1000  # we'll stop after we find 2000 relevant alerts, more than enough for our purposes here

with babamul.AlertConsumer(
    topics=topics,
    offset="earliest",
    auto_commit=False,
    timeout=15,
) as consumer:
    for alert in tqdm(consumer, desc="Filtering relevant alerts"):
        if is_relevant(alert):
            alerts.append(alert)
        if len(alerts) >= limit:
            break

print(f"Fetched {len(alerts)} alerts.")

In [ ]:
# We deduplicate by objectId, keeping the most recent alert for each
# scan for these alerts
alerts_to_scan = {}
for a in alerts:
    if (
        a.objectId not in alerts_to_scan
        or a.candidate.jd > alerts_to_scan[a.objectId].candidate.jd
    ):
        alerts_to_scan[a.objectId] = a
alerts_to_scan = list(alerts_to_scan.values())
print(f"After deduplication, {len(alerts_to_scan)} unique alerts remain.")

In [ ]:
# view objects

from babamul import jupyter as jp
# babamul.jupyter.scan_alerts(
jp.scan_alerts(
    sorted(
        alerts_to_scan, key=lambda a: a.candidate.magpsf, reverse=True
    )
)